In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from delta.tables import DeltaTable

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_github_events"
SILVER_WATCH_TABLE = f"{CATALOG}.{SCHEMA}.silver_watch_events"

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.oss_ecosystem.silver_watch_events (
    event_id STRING,
    event_ts TIMESTAMP,
    actor_id BIGINT,
    repo_id BIGINT,
    action STRING,
    source_file STRING,
    source_hour TIMESTAMP,
    ingested_at TIMESTAMP
)
USING DELTA
""")

In [0]:
watch_payload_schema = StructType([
    StructField("action", StringType(), True)
])

In [0]:
watch_bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "WatchEvent")
)

print(
    "WatchEvent rows in Bronze:",
    watch_bronze_df.count()
)

In [0]:
existing_watch_ids = (
    spark.table(SILVER_WATCH_TABLE)
    .select("event_id")
)

new_watch_bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "WatchEvent")
    .join(
        existing_watch_ids,
        on="event_id",
        how="left_anti"
    )
)

print(
    "New WatchEvents to transform:",
    new_watch_bronze_df.count()
)

In [0]:
silver_watch_df = (
    new_watch_bronze_df
    .withColumn(
        "payload_parsed",
        F.from_json(
            F.col("payload_json"),
            watch_payload_schema
        )
    )
    .select(
        "event_id",

        F.to_timestamp("created_at")
            .alias("event_ts"),

        F.get_json_object("actor_json", "$.id")
            .cast("long")
            .alias("actor_id"),

        F.get_json_object("repo_json", "$.id")
            .cast("long")
            .alias("repo_id"),

        F.col("payload_parsed.action")
            .alias("action"),

        "source_file",
        "source_hour",
        "ingested_at"
    )
)

In [0]:
print(
    "Silver Watch rows:",
    silver_watch_df.count()
)

silver_watch_df.select(
    F.sum(F.col("event_id").isNull().cast("int"))
        .alias("null_event_id"),

    F.sum(F.col("actor_id").isNull().cast("int"))
        .alias("null_actor_id"),

    F.sum(F.col("repo_id").isNull().cast("int"))
        .alias("null_repo_id"),

    F.sum(F.col("action").isNull().cast("int"))
        .alias("null_action")
).show()

In [0]:
duplicate_count = (
    silver_watch_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate WatchEvent IDs:",
    duplicate_count
)

In [0]:
if not spark.catalog.tableExists(SILVER_WATCH_TABLE):
    (
        silver_watch_df.limit(0)
        .write
        .format("delta")
        .saveAsTable(SILVER_WATCH_TABLE)
    )

silver_watch_delta = DeltaTable.forName(
    spark,
    SILVER_WATCH_TABLE
)

(
    silver_watch_delta.alias("t")
    .merge(
        silver_watch_df.alias("s"),
        "t.event_id = s.event_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver Watch MERGE completed.")

In [0]:
print(
    "Silver Watch table rows:",
    spark.table(SILVER_WATCH_TABLE).count()
)

print(
    "Duplicate WatchEvent IDs:",
    spark.table(SILVER_WATCH_TABLE)
        .groupBy("event_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)